# Remote server smoke tests

This notebook queries the running MaterialX remote server endpoints and prints their JSON responses.
Adjust `BASE_URL` if your server runs on a different host/port.

In [41]:
# Configure the base URL for the running remote server
BASE_URL = 'http://127.0.0.1:2907'  # adjust if needed
import requests, json
from pprint import pprint
def show_response(r):
    print(r.status_code, r.headers.get('Content-Type'))
    try: pprint(r.json())
    except Exception: print(r.text)

In [ ]:
# Test: fetch vertex stage, add comment, POST only vertex back, then verify
r = requests.get(f'{BASE_URL}/shader')
print('GET /shader ->', r.status_code)
data = r.json() if r.status_code==200 else {}
stages = data.get('stages', {})
vertex = stages.get('vertex', '')
fragment = stages.get('fragment', '')
if not vertex:
    print('No vertex stage available to test')
else:
    # Prepend a simple comment to the vertex shader source to test override
    commented_vertex = '/* override-test: added comment */' + vertex
    payload = {'vertex': commented_vertex}
    r2 = requests.post(f'{BASE_URL}/shader', json=payload)
    print('POST /shader ->', r2.status_code)
    try:
        pprint(r2.json())
    except Exception:
        print(r2.status_code, r2.text)
    # Verify override is returned by GET /shader
    r3 = requests.get(f'{BASE_URL}/shader')
    
    print('GET /shader (verify) ->', r3.status_code)
    try:
        resp = r3.json(); pprint(resp)
        v2 = resp.get('stages', {}).get('vertex', '')
        if v2.startswith('/* override-test: added comment */'):
            print('Vertex override verified')
        else:
            print('Vertex override not present')
    except Exception:
        print('Verification failed', r3.status_code, r3.text)

In [ ]:
# Test: fetch fragment stage, add comment, POST only fragment back, then verify
r = requests.get(f'{BASE_URL}/shader')
print('GET /shader ->', r.status_code)
data = r.json() if r.status_code==200 else {}
stages = data.get('stages', {})
vertex = stages.get('vertex', '')
fragment = stages.get('fragment', '')
if not fragment:
    print('No fragment stage available to test')
else:
    # Prepend a simple comment to the fragment shader source to test override
    commented_fragment = '/* override-test: fragment comment */' + fragment
    payload = {'fragment': commented_fragment}
    r2 = requests.post(f'{BASE_URL}/shader', json=payload)
    print('POST /shader ->', r2.status_code)
    try:
        pprint(r2.json())
    except Exception:
        print(r2.status_code, r2.text)
    # Verify override is returned by GET /shader
    r3 = requests.get(f'{BASE_URL}/shader')
    print('GET /shader (verify) ->', r3.status_code)
    try:
        resp = r3.json(); pprint(resp)
        f2 = resp.get('stages', {}).get('fragment', '')
        if f2.startswith('/* override-test: fragment comment */'):
            print('Fragment override verified')
        else:
            print('Fragment override not present')
    except Exception:
        print('Verification failed', r3.status_code, r3.text)

In [ ]:
# Get shader for currently selected material
r = requests.get(f'{BASE_URL}/shader')
print('GET /shader ->', r.status_code)
show_response(r)

In [ ]:
# Test: Post a simple fragment shader that forces red output and verify override
# This shader writes red to the first output (out1)
red_fragment = ('/* override-test: force red */\n'
                '#version 400\n'
                '\n'
                'out vec4 out1;\n'
                'void main() { out1 = vec4(1.0, 0.0, 0.0, 1.0); }')
payload = {'fragment': red_fragment}
r = requests.post(f'{BASE_URL}/shader', json=payload)
print('POST /shader (force red) ->', r.status_code)
show_response(r)
# Verify GET returns the override (merged), and check fragment stage starts with our marker
r2 = requests.get(f'{BASE_URL}/shader')
print('GET /shader (verify red) ->', r2.status_code)
try:
    data = r2.json() if r2.status_code==200 else {}
    frag = data.get('stages', {}).get('fragment', '')
    if frag.startswith('/* override-test: force red */'):
        print('Red fragment override present')
    else:
        print('Red fragment override NOT present')
except Exception:
    print('Failed to verify GET /shader', r2.status_code, r2.text)

In [ ]:
# Test: POST /render (compile, render N frames) — authoritative decode using server metadata
import json, re
from pprint import pprint
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# Request parameters
payload = {'frames': 1, 'width': 128, 'height': 128, 'warmup': 10}

# POST to render endpoint
r = requests.post(f'{BASE_URL}/render', json=payload)
print('POST /render ->', r.status_code, r.headers.get('Content-Type'))
if r.status_code != 200:
    try:
        pprint(r.json())
    except Exception:
        print(r.status_code, r.text)
else:
    # Extract multipart boundary
    ctype = r.headers.get('Content-Type', '')
    m = re.search('boundary=(.+)', ctype)
    if not m:
        raise RuntimeError(f'Missing boundary in Content-Type: {ctype}')
    boundary = m.group(1).encode('utf-8')

    parts = r.content.split(b'--' + boundary)

    # Parse parts: find JSON metadata and collect binary frame parts in order
    meta = None
    binary_parts = []
    for p in parts:
        p = p.strip()
        if not p or p == b'--':
            continue
        header_end = p.find(b'\r\n\r\n')
        if header_end == -1:
            continue
        header_block = p[:header_end].decode('utf-8', errors='ignore')
        body = p[header_end+4:]
        headers = {}
        for line in header_block.split('\r\n'):
            if ':' in line:
                k,v = line.split(':',1)
                headers[k.strip().lower()] = v.strip()
        ctype_part = headers.get('content-type','')
        if ctype_part.startswith('application/json') and meta is None:
            meta = json.loads(body.decode('utf-8'))
        elif ctype_part.startswith('application/octet-stream'):
            binary_parts.append(body)

    if meta is None:
        raise RuntimeError('No JSON metadata part found')

    # Print top-level status and dims
    pprint({'status': meta.get('status'), 'frames': meta.get('frames'), 'width': meta.get('width'), 'height': meta.get('height')})

    # Print render timings (per-frame and aggregate) if present
    timings = meta.get('timings_ms', [])

    frames_info = meta.get('frames_info', [])
    if not frames_info:
        raise RuntimeError('No frames_info in metadata')

    # Decode each frame using authoritative frames_info
    for i, fi in enumerate(frames_info):
        # Include per-frame timing (if available) in the header
        timing_str = ''
        if timings and i < len(timings):
            timing_str = f' - {timings[i]:.3f} ms'

        print('\nFrame', i, timing_str)
        expected_bytes = int(fi.get('byteLength'))
        dtype_str = fi.get('dtype')
        channels = int(fi.get('channels'))
        w = int(fi.get('width'))
        h = int(fi.get('height'))

        if i >= len(binary_parts):
            print('Missing binary part for frame', i)
            continue
        body = binary_parts[i]
        if len(body) != expected_bytes:
            print(f'Warning: binary part size {len(body)} != expected {expected_bytes}')

        # Supported dtype mapping (server currently uses float32)
        if dtype_str == 'float32':
            dtype = np.float32
        elif dtype_str == 'uint8':
            dtype = np.uint8
        elif dtype_str == 'uint16':
            dtype = np.uint16
        else:
            raise RuntimeError('Unsupported dtype: ' + str(dtype_str))

        # Convert and reshape
        arr = np.frombuffer(body, dtype=dtype)
        arr = arr.reshape((h, w, channels))

        # For float data, clamp to [0,1] and convert for display
        if dtype == np.float32:
            img_rgb = arr[:, :, :3] if channels >= 3 else np.repeat(arr[:, :, 0:1], 3, axis=2)
            img_rgb = np.clip(img_rgb, 0.0, 1.0)
            disp = (img_rgb[::-1] * 255.0).astype(np.uint8)  # flip vertical
            img = Image.fromarray(disp, 'RGB')
        else:
            # uint8/uint16 path
            if dtype == np.uint16:
                disp = (arr >> 8).astype(np.uint8)
            else:
                disp = arr
            disp = disp[::-1]  # flip vertical
            mode = 'RGB' if channels == 3 else 'RGBA'
            img = Image.fromarray(disp, mode)

        plt.figure(figsize=(4,4))
        plt.title(f'Frame {i} ({w}x{h}, {dtype_str}, {channels}ch){timing_str}')
        plt.imshow(img)
        plt.axis('off')
        plt.show()


In [42]:
# Health and materials tests
r = requests.get(f'{BASE_URL}/health')
print('GET /health ->', r.status_code)
show_response(r)

r = requests.get(f'{BASE_URL}/materials')
print('GET /materials ->', r.status_code)
show_response(r)

# Optionally select the first material from the catalog
try:
    mats = r.json() if r.status_code==200 else []
except Exception:
    mats = []
if isinstance(mats, list) and mats:
    name = mats[0].get('name')
    if name:
        print('Selecting material:', name)
        r2 = requests.post(f'{BASE_URL}/materials/select', json={'name': name})
        print('POST /materials/select ->', r2.status_code)
        show_response(r2)


GET /health -> 200
200 application/json
{'status': 'ok'}
GET /materials -> 200
200 application/json
[{'filePath': 'C:\\Users\\Dmitrii\\Documents\\MaterialX\\resources/Materials\\a_wood\\TH_Wood_Table.mtlx',
  'name': 'TH_Wood_Table.mtlx',
  'verified': False},
 {'filePath': 'C:\\Users\\Dmitrii\\Documents\\MaterialX\\resources/Materials\\Examples\\DisneyPrincipled\\disney_principled_carpaint.mtlx',
  'name': 'disney_principled_carpaint.mtlx',
  'verified': False},
 {'filePath': 'C:\\Users\\Dmitrii\\Documents\\MaterialX\\resources/Materials\\Examples\\DisneyPrincipled\\disney_principled_default.mtlx',
  'name': 'disney_principled_default.mtlx',
  'verified': False},
 {'filePath': 'C:\\Users\\Dmitrii\\Documents\\MaterialX\\resources/Materials\\Examples\\DisneyPrincipled\\disney_principled_glass.mtlx',
  'name': 'disney_principled_glass.mtlx',
  'verified': False},
 {'filePath': 'C:\\Users\\Dmitrii\\Documents\\MaterialX\\resources/Materials\\Examples\\DisneyPrincipled\\disney_principled_go

In [43]:
# Geometry, lights, camera tests
r = requests.get(f'{BASE_URL}/geometry')
print('GET /geometry ->', r.status_code)
show_response(r)

# Lights
r = requests.get(f'{BASE_URL}/lights')
print('GET /lights ->', r.status_code)
show_response(r)

# Camera
r = requests.get(f'{BASE_URL}/camera')
print('GET /camera ->', r.status_code)
show_response(r)


GET /geometry -> 200
200 application/json
{'active': 'Preview_Mesh', 'geometry': ['Preview_Mesh', 'Calibration_Mesh']}
GET /lights -> 200
200 application/json
{'envLightIntensity': 1.0,
 'envRadianceFilename': 'C:\\Users\\Dmitrii\\Documents\\MaterialX\\resources\\Lights\\san_giuseppe_bridge_split.hdr',
 'lightRotation': 0.0}
GET /camera -> 200
200 application/json
{'position': [0.0, 0.0, 5.0],
 'target': [0.0, 0.0, 0.0],
 'viewAngle': 45.0,
 'zoom': 1.0}


In [44]:
# GET /uniforms: list public uniforms and UI metadata
r = requests.get(f'{BASE_URL}/uniforms')
print('GET /uniforms ->', r.status_code)
if r.status_code == 200:
    data = r.json()
    pprint(data)
else:
    show_response(r)

# If there are uniforms, print the first one for manual inspection
if r.status_code == 200:
    u = data.get('uniforms', [])
    if u:
        print('\nFirst uniform:')
        pprint(u[0])


GET /uniforms -> 200
{'uniforms': [{'name': 'node_multiply_6_in2',
               'path': 'NG_TH_Wood_Table/uv/value',
               'type': 'float',
               'value': '2',
               'variable': 'node_multiply_6_in2'},
              {'name': 'node_image_color3_0_file',
               'path': 'NG_TH_Wood_Table/node_image_color3_0/file',
               'type': 'filename',
               'value': 'textures/TH_Wood_Table_baseColor.png',
               'variable': 'node_image_color3_0_file'},
              {'name': 'node_image_vector3_9_file',
               'path': 'NG_TH_Wood_Table/node_image_vector3_9/file',
               'type': 'filename',
               'value': 'textures/TH_Wood_Table_roughness.png',
               'variable': 'node_image_vector3_9_file'},
              {'name': 'node_image_vector3_7_file',
               'path': 'NG_TH_Wood_Table/node_image_vector3_7/file',
               'type': 'filename',
               'value': 'textures/TH_Wood_Table_normal.png',
 

In [45]:
# POST /uniforms: update a uniform (example)
# Prepare a small payload to update the first uniform if available
if 'data' in globals() and data.get('uniforms'):
    first = data['uniforms'][0]
    path = first.get('path') or first.get('name') or first.get('variable')
    print('Attempting to set uniform', path)
    # For the example, toggle boolean or set numeric to a small value; adapt per-type
    example_value = None
    t = first.get('type','')
    if 'float' in t.lower() or t.lower()=='float':
        example_value = '0.5'
    elif 'integer' in t.lower() or t.lower()=='int':
        example_value = '1'
    elif 'boolean' in t.lower() or t.lower()=='bool':
        example_value = 'true'
    elif 'color' in t.lower() or 'vector' in t.lower():
        example_value = '0.5,0.5,0.5'
    elif 'string' in t.lower():
        example_value = 'example'
    else:
        example_value = first.get('value','')

    payload = {'uniforms': [ {'path': path, 'value': example_value} ] }
    r = requests.post(f'{BASE_URL}/uniforms', json=payload)
    print('POST /uniforms ->', r.status_code)
    show_response(r)
else:
    print('No uniform data available to POST example')


Attempting to set uniform NG_TH_Wood_Table/uv/value
POST /uniforms -> 200
200 application/json
{'results': [{'path': 'NG_TH_Wood_Table/uv/value', 'status': 'ok'}]}


In [46]:
# POST /shader/reset: restore canonical shader package from selected material
r = requests.post(f'{BASE_URL}/shader/reset')
print('POST /shader/reset ->', r.status_code)
show_response(r)

# Verify that GET /shader returns generated stages (no overrides) or the canonical package stored
r = requests.get(f'{BASE_URL}/shader')
print('GET /shader ->', r.status_code)
show_response(r)


POST /shader/reset -> 200
200 application/json
{'status': 'ok'}
GET /shader -> 200
200 application/json
{'stages': {'fragment': '#version 400\n'
                        '\n'
                        '\n'
                        'struct BSDF { vec3 response; vec3 throughput; };\n'
                        '#define EDF vec3\n'
                        'struct surfaceshader { vec3 color; vec3 transparency; '
                        '};\n'
                        'struct volumeshader { vec3 color; vec3 transparency; '
                        '};\n'
                        'struct displacementshader { vec3 offset; float scale; '
                        '};\n'
                        'struct lightshader { vec3 intensity; vec3 direction; '
                        '};\n'
                        '#define material surfaceshader\n'
                        '\n'
                        '// Uniform block: PrivateUniforms\n'
                        'uniform sampler2D u_shadowMap;\n'
                    